# Explore cleaned sensor data

Load `dbo.CleanedSensorData` from the SQL Server instance and database configured in the repository's `.env`. This notebook uses `src.database.get_connection()` and performs read-only queries.

Before running, install `notebooks/requirements.txt`, configure `.env` from `.env.example`, and select the corresponding Python kernel. Microsoft ODBC Driver 18 for SQL Server must also be installed on the machine running the kernel.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style="whitegrid")
from dotenv import load_dotenv

# Support launching Jupyter from either the repository root or notebooks/.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "database.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch Jupyter from the repository root or notebooks directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")
from src.database import get_connection, close_connection


## Select the data window

The default loads all rows in timestamp order. Set either boundary to an ISO timestamp to limit memory use. The start is inclusive and the end is exclusive; leave a boundary as `None` to omit it.


In [ ]:
START_TIME = None  # Example: "2026-03-24 00:00:00"
END_TIME = None    # Example: "2026-04-24 00:00:00"

start = pd.Timestamp(START_TIME).to_pydatetime() if START_TIME is not None else None
end = pd.Timestamp(END_TIME).to_pydatetime() if END_TIME is not None else None
if start is not None and end is not None and start >= end:
    raise ValueError("START_TIME must be earlier than END_TIME.")

conditions = []
parameters = []
if start is not None:
    conditions.append("[datetime] >= ?")
    parameters.append(start)
if end is not None:
    conditions.append("[datetime] < ?")
    parameters.append(end)

query = "SELECT * FROM [dbo].[CleanedSensorData]"
if conditions:
    query += " WHERE " + " AND ".join(conditions)
query += " ORDER BY [datetime]"


In [ ]:
connection = get_connection()
try:
    cursor = connection.cursor()
    try:
        cursor.execute(query, *parameters)
        columns = [column[0] for column in cursor.description]
        # Convert ODBC rows to tuples for a consistent pandas tabular shape.
        df = pd.DataFrame.from_records(
            [tuple(row) for row in cursor], columns=columns
        )
    finally:
        cursor.close()
finally:
    close_connection(connection)

# SQL NULL values remain missing; no imputation is applied.
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.set_index("datetime").sort_index()
print(f"Loaded {len(df):,} rows and {len(df.columns)} sensor columns.")
df.head()


## Inspect the dataset

Review the timestamp range, column types, missing readings, and summary statistics before choosing features or a training split. An empty result can indicate that the selected date window contains no imported data.


In [ ]:
df.info()
print("First timestamp:", df.index.min())
print("Last timestamp:", df.index.max())
print("Duplicate timestamps:", df.index.duplicated().sum())


In [ ]:
missing = pd.DataFrame({
    "missing_rows": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100),
})
missing.sort_values("missing_rows", ascending=False)


In [ ]:
df.describe()


## Null values per sensor

The chart includes every sensor, including columns with zero missing values. Labels show the exact null count and percentage of loaded rows. These counts describe SQL NULL readings in existing rows; timestamps absent from the table are not counted.


In [ ]:
if df.empty or len(df.columns) == 0:
    print("No sensor rows to summarize. Check the selected data window.")
else:
    null_summary = missing.sort_values("missing_rows", ascending=False)
    fig, ax = plt.subplots(figsize=(12, max(4, 0.35 * len(null_summary))))
    sns.barplot(
        x=null_summary["missing_rows"].to_numpy(),
        y=null_summary.index.tolist(),
        color="steelblue", ax=ax,
    )
    for position, row in enumerate(null_summary.itertuples()):
        ax.annotate(
            f"{row.missing_rows:,} ({row.missing_percent:.1f}%)",
            (row.missing_rows, position), xytext=(5, 0),
            textcoords="offset points", va="center",
        )
    ax.set_xlim(0, max(1, null_summary["missing_rows"].max()) * 1.3)
    ax.set(title=f"Null readings per sensor ({len(df):,} rows)",
           xlabel="Number of null values", ylabel="Sensor")
    fig.tight_layout()
    plt.show()


## Sensor readings over time

Each sensor gets its own plot and value scale, with a shared time axis. By default, all sensor columns are plotted. Set `PLOT_COLUMNS` to a list such as `["A1_Qin", "A1_COD_in"]` to focus on selected sensors.

These are the original readings without averaging or imputation. Null readings break the line; missing timestamps can still be connected across a gap. Markers make individual observations visible, including isolated readings. An entirely null sensor has an empty plot.


In [ ]:
PLOT_COLUMNS = None  # None plots all sensors; or provide a list of column names.

plot_columns = list(df.columns) if PLOT_COLUMNS is None else list(PLOT_COLUMNS)
unknown_columns = [column for column in plot_columns if column not in df.columns]
if unknown_columns:
    raise ValueError(f"Unknown sensor columns: {unknown_columns}")

if df.empty or not plot_columns:
    print("No readings or sensor columns to plot. Check the data window and PLOT_COLUMNS.")
else:
    # Convert all-null object columns to numeric while retaining their missing values.
    plot_data = df[plot_columns].apply(pd.to_numeric, errors="raise")
    fig, axes = plt.subplots(
        len(plot_columns), 1, sharex=True,
        figsize=(14, max(3, 2.5 * len(plot_columns))), squeeze=False,
    )
    for ax, column in zip(axes[:, 0], plot_columns):
        ax.plot(plot_data.index, plot_data[column], linewidth=1,
                marker=".", markersize=2, color="steelblue")
        ax.set_title(column, loc="left")
        ax.set_ylabel("Value")
        if plot_data[column].isna().all():
            ax.text(0.5, 0.5, "All readings are null", transform=ax.transAxes,
                    ha="center", va="center")
    locator = mdates.AutoDateLocator()
    axes[-1, 0].xaxis.set_major_locator(locator)
    axes[-1, 0].xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    axes[-1, 0].set_xlabel("Timestamp")
    fig.suptitle("Cleaned sensor readings over time")
    fig.tight_layout(rect=(0, 0, 1, 0.98))
    plt.show()
